# Diffusion Model for Financial Time Series

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU**

| Cell | What it does |
|------|--------------|
| Setup | Mount Drive, set paths, check GPU |
| Config | All hyperparameters in one place |
| Train | Train the diffusion model |
| Sample | Generate synthetic time series from a checkpoint |
| Evaluate | Compute discriminative/predictive/VDS/FDDS scores |

## 1. Setup

In [1]:
# ── Check GPU ────────────────────────────────────────────────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory  : {mem_gb:.1f} GB")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
Memory  : 42.4 GB


In [2]:
# ── Clone repo from GitHub ────────────────────────────────────────────────────
import os
REPO_DIR = '/content/DiffusionModelTimeSeries'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Ardameliksah/DiffusionModelTimeSeries.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

Repo already cloned, pulling latest...
Already up to date.


In [3]:
import os, sys
from pathlib import Path

REPO_PATH = REPO_DIR  # set by the clone cell above

assert Path(REPO_PATH).exists(), f"Folder not found: {REPO_PATH}"

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"Working directory: {os.getcwd()}")
print("Files found:", [f for f in os.listdir() if f.endswith('.py')])

Working directory: /content/DiffusionModelTimeSeries
Files found: ['evaluate_unified.py', 'visualize_samples.py', 'sanity.py', 'quick_range_check.py', 'sample_unified.py', '__init__.py', 'eval_metrics.py', 'input_test.py', 'train_with_mode.py', 'diagnose_x0_range.py', 'diag_t_bins.py']


In [4]:
# ── Install any missing packages ────────────────────────────────────────────
# Colab already has torch, numpy, pandas, matplotlib, scikit-learn, scipy,
# seaborn, tqdm, pillow. Only install what might be missing.
!pip install -q --upgrade pip
!pip install -q wandb
!pip show tqdm scikit-learn seaborn | grep -E 'Name|Version'

Name: tqdm
Version: 4.67.3
Name: scikit-learn
Version: 1.6.1
 Name: GCC runtime library
 Version 3.1, 31 March 2009
Name: seaborn
Version: 0.13.2


In [9]:
# ── Weights & Biases login ────────────────────────────────────────────────────
# Option A: set a Colab Secret named WANDB_API_KEY (Secrets panel, left sidebar)
# Option B: enter your key interactively when prompted below
import wandb, os
#wandb apı key:
os.environ["WANDB_API_KEY"] = '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'

_api_key = os.environ.get("WANDB_API_KEY")   # populated by Colab Secrets
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()                              # interactive prompt — paste API key

print("wandb version:", wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb version: 0.26.1


## 2. Configuration

Edit the values below before running the Train / Sample / Evaluate cells.

In [12]:
import torch

# ── Mode ──────────────────────────────────────────────────────────────────────
MODE      = "image"    # "raw" or "image"
EMBEDDING = "stft"  # "delay" | "patch" | "stft" | "mrti"  (image mode only)

# ── Training ──────────────────────────────────────────────────────────────────
NUM_EPOCHS     = 500
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-4   # None = config default (5e-4 raw, 1e-3 image)
NOISE_SCHEDULE = None   # None | "linear" | "cosine" | "exponential"
NORMALIZATION  = None   # None | "minmax" | "zscore"
HIDDEN_DIM     = 64   # None | 64 (small) | 128 (medium) | 256 (large)
NUM_LAYERS     = 3   # None | 3 (small) | 6 (medium) | 8 (large)
POS_ENC        = None   # None | "learnable" | "fixed"  (raw mode only)
NUM_WORKERS    = 2      # 0 on Windows; 2-4 on Colab (Linux)
SEED           = 42
RESUME_FROM    = None
CHECKPOINT_DIR = None

# ── Image-mode objective (ignored in raw mode) ────────────────────────────────
# Match raw mode exactly:  IMG_PRED_OBJECTIVE="pred_x0"  IMG_LOSS_TYPE="l1"
# Keep DDPM defaults:      IMG_PRED_OBJECTIVE="pred_eps"  IMG_LOSS_TYPE="mse"
IMG_PRED_OBJECTIVE = "pred_x0"  # "pred_eps" | "pred_x0"
IMG_LOSS_TYPE      = "l1"       # "mse" | "l1"

# ── Weights & Biases ─────────────────────────────────────────────────────────
USE_WANDB     = True
WANDB_PROJECT = "diffusion-timeseries"

# ── Inline evaluation metrics (during training) ───────────────────────────────
EVAL_METRICS              = True
EVAL_METRICS_EVERY        = 100
N_METRIC_ITERATIONS_TRAIN = 3
NUM_METRIC_SAMPLES        = 128

# ── Sampling ──────────────────────────────────────────────────────────────────
NUM_SAMPLES     = 256
NUM_STEPS       = 200
ETA             = 0.0
CHECKPOINT_PATH = None

# ── Evaluation ────────────────────────────────────────────────────────────────
N_METRIC_ITERATIONS = 5
COMPUTE_CONTEXT_FID = False

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device  : {DEVICE}")
print(f"Mode    : {MODE}")
print(f"Epochs  : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  Workers: {NUM_WORKERS}")
print(f"W&B     : {'enabled → ' + WANDB_PROJECT if USE_WANDB else 'disabled'}")
print(f"Metrics : {'every ' + str(EVAL_METRICS_EVERY) + ' epochs' if EVAL_METRICS else 'disabled'}")
if MODE == 'image':
    print(f"Embedding  : {EMBEDDING}")
    print(f"Objective  : {IMG_PRED_OBJECTIVE}  |  Loss: {IMG_LOSS_TYPE}")

Device  : cuda
Mode    : image
Epochs  : 500  |  Batch: 64  |  Workers: 2
W&B     : enabled → diffusion-timeseries
Metrics : every 100 epochs
Embedding  : stft
Objective  : pred_x0  |  Loss: l1


## 3. Train

Trains the model and saves checkpoints to `output/checkpoints/` (raw mode) or `output/checkpoints_image/` (image mode).
The best validation-loss checkpoint is always saved as `best_model.pt`.

In [11]:
from train_with_mode import train

train(
    mode=MODE,
    device=DEVICE,
    resume_from=RESUME_FROM,
    embedding=EMBEDDING,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    noise_schedule=NOISE_SCHEDULE,
    checkpoint_dir=CHECKPOINT_DIR,
    normalization=NORMALIZATION,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    seed=SEED,
    pos_enc=POS_ENC,
    lr=LEARNING_RATE,
    num_workers=NUM_WORKERS,
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT,
    eval_metrics=EVAL_METRICS,
    eval_metrics_every=EVAL_METRICS_EVERY,
    n_metric_iterations=N_METRIC_ITERATIONS_TRAIN,
    num_metric_samples=NUM_METRIC_SAMPLES,
    img_pred_objective=IMG_PRED_OBJECTIVE,   # image mode only
    img_loss_type=IMG_LOSS_TYPE,             # image mode only
)

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - IMAGE MODE
{'image': {'embedding_type': 'delay', 'delay': 4, 'embedding_dim': 8, 'patch_size': 4, 'img_size': 8, 'n_fft': 14, 'hop_length': 4, 'num_scales': 3, 'num_periods': 3, 'pad_to_square': True, 'pad_value': 0.0}, 'model': {'hidden_dim': 64, 'num_layers': 3, 'num_heads': 8, 'ff_dim': 256, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'pred_objective': 'pred_x0', 'loss_type': 'l1', 'image_height': 8, 'image_width': 8}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'linear', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 500, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': '/content/DiffusionModelTimeSeries/output/checkpoints_image', 'log_dir': '/content/DiffusionModelTimeSeries/output/logs_image', 'save_every_n_epochs': 50, 'validate_every_n_epochs':

epoch,▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
lr,▇████████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train_loss,█▆▅▄▄▄▃▃▃▃▄▃▃▃▃▂▂▂▂▃▁▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▂▁▁▁
val_loss,█▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_epoch,231
best_val_loss,0.22576
epoch,500
lr,0.0
train_loss,0.22384
val_loss,0.22969



Training complete! (image mode)
Best val loss: 0.2258
Checkpoints saved to: /content/DiffusionModelTimeSeries/output/checkpoints_image


## 4. Sample

Generates `NUM_SAMPLES` synthetic windows from a trained checkpoint and saves them as `.npz` to `output/generated_samples/`.

In [ ]:
from pathlib import Path
from sample_unified import generate, denormalize, save_samples
from config.stocks_config import Config as RawConfig
from config.image_config import ImageVersionConfig
from utils.data_utils import StockDataset, build_scaler

# Auto-detect best_model.pt if not set manually
_ckpt = CHECKPOINT_PATH
if _ckpt is None:
    if MODE == "raw":
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints" / "best_model.pt")
    else:
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints_image" / "best_model.pt")

assert Path(_ckpt).exists(), f"Checkpoint not found: {_ckpt}"
print(f"Loading checkpoint: {_ckpt}")

# Build config
if MODE == "raw":
    _config = RawConfig()
else:
    _config = ImageVersionConfig()
    _config.image.embedding_type = EMBEDDING

if NORMALIZATION is not None:
    _config.data.neg_one_to_one = (NORMALIZATION == "minmax")

# Generate
samples = generate(
    checkpoint_path=_ckpt,
    mode=MODE,
    config=_config,
    num_samples=NUM_SAMPLES,
    num_steps=NUM_STEPS,
    eta=ETA,
    device=DEVICE,
)

# Denormalize back to original price scale
dataset = StockDataset(
    csv_path=_config.data.data_path,
    scaler=build_scaler(_config.data.neg_one_to_one),
    window_length=_config.model.sequence_length,
)
samples_denorm = denormalize(samples, dataset)

_emb = EMBEDDING if MODE == "image" else None
save_samples(samples_denorm, _config.sampling.output_dir, MODE, _emb)
print(f"\nFinal shape: {samples_denorm.shape}  (samples, channels, time)")

### Quick visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_show = min(6, samples_denorm.shape[0])
channel_names = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

fig, axes = plt.subplots(n_show, 1, figsize=(12, 2 * n_show))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Plot Close price (channel 3) for readability
    ax.plot(samples_denorm[i, 3, :], linewidth=1)
    ax.set_ylabel("Close", fontsize=8)
    ax.set_title(f"Sample {i+1}", fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle(f"Generated Samples — {MODE} mode", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Evaluate

Computes:
- **Discriminative score** — GRU classifier real vs synthetic (target: 0.0, test acc: 0.5)
- **Predictive MAE** — train-on-fake / test-on-real (lower = better)
- **VDS** — KL divergence of value distributions (lower = better)
- **FDDS** — KL divergence of cross-correlation distributions (lower = better)
- **Correlational score** — |CACF_fake − CACF_real| / 10 (lower = better)

Results and a comparison plot are saved to `output/` automatically.

In [ ]:
from evaluate_unified import evaluate

evaluate(
    mode=MODE,
    checkpoint_path=_ckpt,
    device=DEVICE,
    num_samples=NUM_SAMPLES,
    output_dir=None,               # None = config default
    n_metric_iterations=N_METRIC_ITERATIONS,
    compute_context_fid=COMPUTE_CONTEXT_FID,
    normalization=NORMALIZATION,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    seed=SEED,
    pos_enc=POS_ENC,
    use_wandb=USE_WANDB,           # ← logs all eval metrics to W&B
    wandb_project=WANDB_PROJECT,
)

## (Optional) Copy outputs back to Drive

Colab runtimes are ephemeral — outputs written inside `/content/` are lost when the session ends.  
If your `REPO_PATH` already points to Drive, checkpoints are already persistent. Otherwise run the cell below.

In [ ]:
# Only needed if REPO_PATH is NOT on Drive (e.g. you cloned to /content/)
import shutil

DRIVE_BACKUP = '/content/drive/MyDrive/TezBaselines/MyCode/output'
LOCAL_OUTPUT = str(Path(REPO_PATH) / 'output')

shutil.copytree(LOCAL_OUTPUT, DRIVE_BACKUP, dirs_exist_ok=True)
print(f"Outputs copied to {DRIVE_BACKUP}")